# **imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

Mounted at /content/drive


# **Training model**

In [ ]:
thisrf_df = pd.read_csv('/content/drive/MyDrive/Aerosol/files/aeronet_mcd_tropomi.csv')

In [ ]:
thisrf_df

,AERONET_Site,Date(dd:mm:yyyy),Time(hh:mm:ss),Single_Scattering_Albedo[440nm],Single_Scattering_Albedo[675nm],Single_Scattering_Albedo[870nm],Single_Scattering_Albedo[1020nm],Solar_Zenith_Angle_for_Measurement_Start(Degrees),Coincident_AOD440nm,Latitude(Degrees),...,longitude,CO,AER_AI,NO2,year,month,day,time,datetime,SZA
0,Ussuriysk,20:02:2021,06:27:32,0.7407,0.8166,0.8056,0.7932,68.430923,0.204313,43.7004,...,131.633588,0.037328,0.363277,0.000011,2021.0,2.0,20.0,02:37:39,2021-02-20 02:37:39,55.890479
1,Ussuriysk,25:02:2021,02:26:54,0.8606,0.8517,0.8153,0.8003,54.319945,0.452988,43.7004,...,132.015267,0.051809,0.271295,0.000068,2021.0,2.0,25.0,02:44:32,2021-02-25 02:44:32,53.701662
2,Ussuriysk,25:02:2021,06:51:59,0.8375,0.8360,0.7950,0.7697,70.518812,0.518240,43.7004,...,132.015267,0.051809,0.271295,0.000068,2021.0,2.0,25.0,02:44:32,2021-02-25 02:44:32,53.701662
3,SEDE_BOKER,23:02:2021,06:54:01,0.9505,0.9792,0.9710,0.9716,59.226454,0.330844,30.8550,...,34.458015,0.028672,0.702084,0.000029,2021.0,2.0,23.0,10:08:24,2021-02-23 10:08:24,40.336469
4,SEDE_BOKER,23:02:2021,07:00:35,0.9461,0.9754,0.9634,0.9586,58.092283,0.328916,30.8550,...,34.458015,0.028672,0.702084,0.000029,2021.0,2.0,23.0,10:08:24,2021-02-23 10:08:24,40.336469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9975,Chachoengsao,25:02:2021,08:29:52,0.9349,0.9441,0.9404,0.9363,50.564117,0.809881,13.5000,...,101.225806,0.053277,-0.144082,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613
9976,Chachoengsao,25:02:2021,09:58:56,0.8876,0.8674,0.8492,0.8291,71.008821,0.819742,13.5000,...,101.225806,0.053277,-0.144082,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613
9977,Chachoengsao,25:02:2021,10:24:38,0.9012,0.8745,0.8510,0.8335,77.058777,0.879299,13.5000,...,101.225806,0.053277,-0.144082,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613
9978,Chachoengsao,26:02:2021,09:59:02,0.8711,0.8648,0.8570,0.8461,70.958276,0.545511,13.5000,...,101.161290,0.047642,-0.140685,0.000087,2021.0,2.0,26.0,05:48:37,2021-02-26 05:48:37,22.366182


In [ ]:
thisrf_df['Predicted_Aerosol_Type'] = -1

In [ ]:
thisrf_df

,AERONET_Site,Date(dd:mm:yyyy),Time(hh:mm:ss),Single_Scattering_Albedo[440nm],Single_Scattering_Albedo[675nm],Single_Scattering_Albedo[870nm],Single_Scattering_Albedo[1020nm],Solar_Zenith_Angle_for_Measurement_Start(Degrees),Coincident_AOD440nm,Latitude(Degrees),...,CO,AER_AI,NO2,year,month,day,time,datetime,SZA,Predicted_Aerosol_Type
0,Ussuriysk,20:02:2021,06:27:32,0.7407,0.8166,0.8056,0.7932,68.430923,0.204313,43.7004,...,0.037328,0.363277,0.000011,2021.0,2.0,20.0,02:37:39,2021-02-20 02:37:39,55.890479,-1
1,Ussuriysk,25:02:2021,02:26:54,0.8606,0.8517,0.8153,0.8003,54.319945,0.452988,43.7004,...,0.051809,0.271295,0.000068,2021.0,2.0,25.0,02:44:32,2021-02-25 02:44:32,53.701662,-1
2,Ussuriysk,25:02:2021,06:51:59,0.8375,0.8360,0.7950,0.7697,70.518812,0.518240,43.7004,...,0.051809,0.271295,0.000068,2021.0,2.0,25.0,02:44:32,2021-02-25 02:44:32,53.701662,-1
3,SEDE_BOKER,23:02:2021,06:54:01,0.9505,0.9792,0.9710,0.9716,59.226454,0.330844,30.8550,...,0.028672,0.702084,0.000029,2021.0,2.0,23.0,10:08:24,2021-02-23 10:08:24,40.336469,-1
4,SEDE_BOKER,23:02:2021,07:00:35,0.9461,0.9754,0.9634,0.9586,58.092283,0.328916,30.8550,...,0.028672,0.702084,0.000029,2021.0,2.0,23.0,10:08:24,2021-02-23 10:08:24,40.336469,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9975,Chachoengsao,25:02:2021,08:29:52,0.9349,0.9441,0.9404,0.9363,50.564117,0.809881,13.5000,...,0.053277,-0.144082,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613,-1
9976,Chachoengsao,25:02:2021,09:58:56,0.8876,0.8674,0.8492,0.8291,71.008821,0.819742,13.5000,...,0.053277,-0.144082,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613,-1
9977,Chachoengsao,25:02:2021,10:24:38,0.9012,0.8745,0.8510,0.8335,77.058777,0.879299,13.5000,...,0.053277,-0.144082,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613,-1
9978,Chachoengsao,26:02:2021,09:59:02,0.8711,0.8648,0.8570,0.8461,70.958276,0.545511,13.5000,...,0.047642,-0.140685,0.000087,2021.0,2.0,26.0,05:48:37,2021-02-26 05:48:37,22.366182,-1


In [ ]:
thisrf_df.columns

Index(['AERONET_Site', 'Date(dd:mm:yyyy)', 'Time(hh:mm:ss)',
       'Single_Scattering_Albedo[440nm]', 'Single_Scattering_Albedo[675nm]',
       'Single_Scattering_Albedo[870nm]', 'Single_Scattering_Albedo[1020nm]',
       'Solar_Zenith_Angle_for_Measurement_Start(Degrees)',
       'Coincident_AOD440nm', 'Latitude(Degrees)', 'Longitude(Degrees)',
       'Elevation(m)', 'Datetime', 'Dust_Ratio', 'Aerosol_Type',
       'Dust_Ratio_PLDR', 'Aerosol_Type_PLDR', 'Year', 'LC_IGBP',
       'Pct_Agreement', 'latitude', 'longitude', 'CO', 'AER_AI', 'NO2', 'year',
       'month', 'day', 'time', 'datetime', 'SZA', 'Predicted_Aerosol_Type'],
      dtype='object')

In [ ]:
thisrf_df['4_Aerosol_Type'] = thisrf_df['Aerosol_Type_PLDR']

In [ ]:
thisrf_df['4_Aerosol_Type'] = thisrf_df['4_Aerosol_Type'].replace({4: 3, 5: 6})

In [ ]:
# Mask for rows where the type is 2
mask_2 = thisrf_df['4_Aerosol_Type'] == 2

# SSA column reference
ssa = thisrf_df['Single_Scattering_Albedo[440nm]']

# Apply conditionally
thisrf_df.loc[mask_2 & (ssa > 0.95), '4_Aerosol_Type'] = 3
thisrf_df.loc[mask_2 & (ssa <= 0.95), '4_Aerosol_Type'] = 6

In [ ]:
features = ['LC_IGBP', 'Pct_Agreement', 'CO', 'AER_AI', 'NO2', 'SZA']
target = 'Aerosol_Type_PLDR'

X = thisrf_df[features]
y = thisrf_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.6, random_state=42)
test_indices = X_test.index

model_this = RandomForestClassifier(random_state=42)
model_this.fit(X_train, y_train)

predictions = model_this.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy * 100)

thisrf_df.loc[test_indices, 'Predicted_Aerosol_Type'] = predictions

Accuracy: 69.48897795591182


In [ ]:
thisrf_df['4_Predicted_Aerosol_Type'] = -1

In [ ]:
features = ['LC_IGBP', 'Pct_Agreement', 'CO', 'AER_AI', 'NO2', 'SZA']
target = '4_Aerosol_Type'

X = thisrf_df[features]
y = thisrf_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.6, random_state=42)
test_indices = X_test.index

model_this_4 = RandomForestClassifier(random_state=42)
model_this_4.fit(X_train, y_train)

predictions = model_this_4.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy * 100)

thisrf_df.loc[test_indices, '4_Predicted_Aerosol_Type'] = predictions

Accuracy: 81.2374749498998


In [ ]:
thisrf_df

,AERONET_Site,Date(dd:mm:yyyy),Time(hh:mm:ss),Single_Scattering_Albedo[440nm],Single_Scattering_Albedo[675nm],Single_Scattering_Albedo[870nm],Single_Scattering_Albedo[1020nm],Solar_Zenith_Angle_for_Measurement_Start(Degrees),Coincident_AOD440nm,Latitude(Degrees),...,NO2,year,month,day,time,datetime,SZA,Predicted_Aerosol_Type,4_Aerosol_Type,4_Predicted_Aerosol_Type
0,Ussuriysk,20:02:2021,06:27:32,0.7407,0.8166,0.8056,0.7932,68.430923,0.204313,43.7004,...,0.000011,2021.0,2.0,20.0,02:37:39,2021-02-20 02:37:39,55.890479,2,6,6
1,Ussuriysk,25:02:2021,02:26:54,0.8606,0.8517,0.8153,0.8003,54.319945,0.452988,43.7004,...,0.000068,2021.0,2.0,25.0,02:44:32,2021-02-25 02:44:32,53.701662,-1,6,-1
2,Ussuriysk,25:02:2021,06:51:59,0.8375,0.8360,0.7950,0.7697,70.518812,0.518240,43.7004,...,0.000068,2021.0,2.0,25.0,02:44:32,2021-02-25 02:44:32,53.701662,-1,6,-1
3,SEDE_BOKER,23:02:2021,06:54:01,0.9505,0.9792,0.9710,0.9716,59.226454,0.330844,30.8550,...,0.000029,2021.0,2.0,23.0,10:08:24,2021-02-23 10:08:24,40.336469,1,1,1
4,SEDE_BOKER,23:02:2021,07:00:35,0.9461,0.9754,0.9634,0.9586,58.092283,0.328916,30.8550,...,0.000029,2021.0,2.0,23.0,10:08:24,2021-02-23 10:08:24,40.336469,-1,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9975,Chachoengsao,25:02:2021,08:29:52,0.9349,0.9441,0.9404,0.9363,50.564117,0.809881,13.5000,...,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613,-1,3,-1
9976,Chachoengsao,25:02:2021,09:58:56,0.8876,0.8674,0.8492,0.8291,71.008821,0.819742,13.5000,...,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613,5,6,6
9977,Chachoengsao,25:02:2021,10:24:38,0.9012,0.8745,0.8510,0.8335,77.058777,0.879299,13.5000,...,0.000044,2021.0,2.0,25.0,06:07:32,2021-02-25 06:07:32,24.217613,-1,3,-1
9978,Chachoengsao,26:02:2021,09:59:02,0.8711,0.8648,0.8570,0.8461,70.958276,0.545511,13.5000,...,0.000087,2021.0,2.0,26.0,05:48:37,2021-02-26 05:48:37,22.366182,4,6,3


In [ ]:
choirf_df = pd.read_csv('/content/drive/MyDrive/Aerosol/files/aeronet_mcd_tropomi_myd.csv')

In [ ]:
choirf_df

,AERONET_Site,Date(dd:mm:yyyy),Time(hh:mm:ss),Single_Scattering_Albedo[440nm],Single_Scattering_Albedo[675nm],Single_Scattering_Albedo[870nm],Single_Scattering_Albedo[1020nm],Solar_Zenith_Angle_for_Measurement_Start(Degrees),Coincident_AOD440nm,Latitude(Degrees),...,Date_MYD,Time_MYD,Latitude_MYD,Longitude_MYD,AOD_550nm_MYD,Angstrom_Exponent_MYD,Reflect_412nm_MYD,Reflect_470nm_MYD,Reflect_660nm_MYD,Datetime_MYD
0,Dalanzadgad,16:02:2021,03:28:29,0.9705,0.9666,0.9532,0.9370,61.005613,0.051455,43.577222,...,2021-02-16,05:55:00,25.395027,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00
1,Dalanzadgad,16:02:2021,04:29:05,0.9679,0.9713,0.9677,0.9754,56.849582,0.049154,43.577222,...,2021-02-16,05:55:00,25.395027,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00
2,Dalanzadgad,16:02:2021,06:28:42,0.9333,0.9749,0.9703,0.9525,58.174913,0.053686,43.577222,...,2021-02-16,05:55:00,25.395027,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00
3,Dalanzadgad,16:02:2021,07:29:07,0.9164,0.9281,0.9389,0.8794,63.440228,0.047013,43.577222,...,2021-02-16,05:55:00,25.395027,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00
4,Dalanzadgad,16:02:2021,08:28:49,0.9120,0.9790,0.9743,0.9387,70.866450,0.055759,43.577222,...,2021-02-16,05:55:00,25.395027,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6263,TGF_Tsukuba,27:02:2021,06:56:02,0.9649,0.9534,0.9258,0.9103,72.247716,0.152418,36.113889,...,2021-02-27,04:00:00,35.821190,139.565480,215,1300,1308,1037,837,2021-02-27 04:00:00
6264,AAU_ET,27:02:2021,06:01:52,0.8851,0.9062,0.8794,0.8789,56.430985,0.257068,9.019167,...,2021-02-27,10:25:00,-3.675921,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00
6265,AAU_ET,27:02:2021,06:26:58,0.8878,0.8927,0.8572,0.8408,50.518303,0.278683,9.019167,...,2021-02-27,10:25:00,-3.675921,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00
6266,AAU_ET,27:02:2021,06:49:53,0.8738,0.8837,0.8539,0.8373,45.197167,0.275837,9.019167,...,2021-02-27,10:25:00,-3.675921,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00


In [ ]:
choirf_df['Predicted_Aerosol_Type'] = -1

In [ ]:
choirf_df['4_Aerosol_Type'] = choirf_df['Aerosol_Type_PLDR']

In [ ]:
choirf_df['4_Aerosol_Type'] = choirf_df['4_Aerosol_Type'].replace({4: 3, 5: 6})

In [ ]:
# Mask for rows where the type is 2
mask_2 = choirf_df['4_Aerosol_Type'] == 2

# SSA column reference
ssa = choirf_df['Single_Scattering_Albedo[440nm]']

# Apply conditionally
choirf_df.loc[mask_2 & (ssa > 0.95), '4_Aerosol_Type'] = 3
choirf_df.loc[mask_2 & (ssa <= 0.95), '4_Aerosol_Type'] = 6

In [ ]:
choirf_df.columns

Index(['AERONET_Site', 'Date(dd:mm:yyyy)', 'Time(hh:mm:ss)',
       'Single_Scattering_Albedo[440nm]', 'Single_Scattering_Albedo[675nm]',
       'Single_Scattering_Albedo[870nm]', 'Single_Scattering_Albedo[1020nm]',
       'Solar_Zenith_Angle_for_Measurement_Start(Degrees)',
       'Coincident_AOD440nm', 'Latitude(Degrees)', 'Longitude(Degrees)',
       'Elevation(m)', 'Datetime', 'Dust_Ratio', 'Aerosol_Type',
       'Dust_Ratio_PLDR', 'Aerosol_Type_PLDR', 'Year', 'LC_IGBP',
       'Pct_Agreement', 'latitude', 'longitude', 'CO', 'AER_AI', 'NO2', 'year',
       'month', 'day', 'time', 'datetime', 'SZA', 'Date', 'Date_MYD',
       'Time_MYD', 'Latitude_MYD', 'Longitude_MYD', 'AOD_550nm_MYD',
       'Angstrom_Exponent_MYD', 'Reflect_412nm_MYD', 'Reflect_470nm_MYD',
       'Reflect_660nm_MYD', 'Datetime_MYD', 'Predicted_Aerosol_Type',
       '4_Aerosol_Type'],
      dtype='object')

In [ ]:
features = ['AOD_550nm_MYD', 'Angstrom_Exponent_MYD', 'Reflect_412nm_MYD',
       'Reflect_470nm_MYD', 'Reflect_660nm_MYD', 'LC_IGBP', 'Pct_Agreement', 'CO', 'AER_AI', 'NO2', 'SZA']
target = 'Aerosol_Type_PLDR'

X = choirf_df[features]
y = choirf_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.6, random_state=42)
test_indices = X_test.index

model_choi = RandomForestClassifier(random_state=42)
model_choi.fit(X_train, y_train)

predictions = model_choi.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy*100)

choirf_df.loc[test_indices, 'Predicted_Aerosol_Type'] = predictions

Accuracy: 72.32854864433811


In [ ]:
choirf_df['4_Predicted_Aerosol_Type'] = -1

In [ ]:
choirf_df

,AERONET_Site,Date(dd:mm:yyyy),Time(hh:mm:ss),Single_Scattering_Albedo[440nm],Single_Scattering_Albedo[675nm],Single_Scattering_Albedo[870nm],Single_Scattering_Albedo[1020nm],Solar_Zenith_Angle_for_Measurement_Start(Degrees),Coincident_AOD440nm,Latitude(Degrees),...,Longitude_MYD,AOD_550nm_MYD,Angstrom_Exponent_MYD,Reflect_412nm_MYD,Reflect_470nm_MYD,Reflect_660nm_MYD,Datetime_MYD,Predicted_Aerosol_Type,4_Aerosol_Type,4_Predicted_Aerosol_Type
0,Dalanzadgad,16:02:2021,03:28:29,0.9705,0.9666,0.9532,0.9370,61.005613,0.051455,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,1,-1
1,Dalanzadgad,16:02:2021,04:29:05,0.9679,0.9713,0.9677,0.9754,56.849582,0.049154,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,3,-1
2,Dalanzadgad,16:02:2021,06:28:42,0.9333,0.9749,0.9703,0.9525,58.174913,0.053686,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,1,-1
3,Dalanzadgad,16:02:2021,07:29:07,0.9164,0.9281,0.9389,0.8794,63.440228,0.047013,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,6,-1
4,Dalanzadgad,16:02:2021,08:28:49,0.9120,0.9790,0.9743,0.9387,70.866450,0.055759,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6263,TGF_Tsukuba,27:02:2021,06:56:02,0.9649,0.9534,0.9258,0.9103,72.247716,0.152418,36.113889,...,139.565480,215,1300,1308,1037,837,2021-02-27 04:00:00,1,1,-1
6264,AAU_ET,27:02:2021,06:01:52,0.8851,0.9062,0.8794,0.8789,56.430985,0.257068,9.019167,...,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00,2,6,-1
6265,AAU_ET,27:02:2021,06:26:58,0.8878,0.8927,0.8572,0.8408,50.518303,0.278683,9.019167,...,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00,2,6,-1
6266,AAU_ET,27:02:2021,06:49:53,0.8738,0.8837,0.8539,0.8373,45.197167,0.275837,9.019167,...,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00,2,6,-1


In [ ]:
features = ['AOD_550nm_MYD', 'Angstrom_Exponent_MYD', 'Reflect_412nm_MYD',
       'Reflect_470nm_MYD', 'Reflect_660nm_MYD', 'LC_IGBP', 'Pct_Agreement', 'CO', 'AER_AI', 'NO2', 'SZA']
target = '4_Aerosol_Type'

X = choirf_df[features]
y = choirf_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.6, random_state=42)
test_indices = X_test.index

model_choi_4 = RandomForestClassifier(random_state=42)
model_choi_4.fit(X_train, y_train)

predictions = model_choi_4.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy*100)

choirf_df.loc[test_indices, '4_Predicted_Aerosol_Type'] = predictions

Accuracy: 82.41626794258373


In [ ]:
choirf_df

,AERONET_Site,Date(dd:mm:yyyy),Time(hh:mm:ss),Single_Scattering_Albedo[440nm],Single_Scattering_Albedo[675nm],Single_Scattering_Albedo[870nm],Single_Scattering_Albedo[1020nm],Solar_Zenith_Angle_for_Measurement_Start(Degrees),Coincident_AOD440nm,Latitude(Degrees),...,Longitude_MYD,AOD_550nm_MYD,Angstrom_Exponent_MYD,Reflect_412nm_MYD,Reflect_470nm_MYD,Reflect_660nm_MYD,Datetime_MYD,Predicted_Aerosol_Type,4_Aerosol_Type,4_Predicted_Aerosol_Type
0,Dalanzadgad,16:02:2021,03:28:29,0.9705,0.9666,0.9532,0.9370,61.005613,0.051455,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,1,-1
1,Dalanzadgad,16:02:2021,04:29:05,0.9679,0.9713,0.9677,0.9754,56.849582,0.049154,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,3,-1
2,Dalanzadgad,16:02:2021,06:28:42,0.9333,0.9749,0.9703,0.9525,58.174913,0.053686,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,1,-1
3,Dalanzadgad,16:02:2021,07:29:07,0.9164,0.9281,0.9389,0.8794,63.440228,0.047013,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,6,-1
4,Dalanzadgad,16:02:2021,08:28:49,0.9120,0.9790,0.9743,0.9387,70.866450,0.055759,43.577222,...,102.462654,138,1500,1254,941,622,2021-02-16 05:55:00,-1,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6263,TGF_Tsukuba,27:02:2021,06:56:02,0.9649,0.9534,0.9258,0.9103,72.247716,0.152418,36.113889,...,139.565480,215,1300,1308,1037,837,2021-02-27 04:00:00,1,1,1
6264,AAU_ET,27:02:2021,06:01:52,0.8851,0.9062,0.8794,0.8789,56.430985,0.257068,9.019167,...,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00,2,6,6
6265,AAU_ET,27:02:2021,06:26:58,0.8878,0.8927,0.8572,0.8408,50.518303,0.278683,9.019167,...,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00,2,6,6
6266,AAU_ET,27:02:2021,06:49:53,0.8738,0.8837,0.8539,0.8373,45.197167,0.275837,9.019167,...,39.693710,133,1250,1595,1244,947,2021-02-27 10:25:00,2,6,6


In [ ]:
drive.mount('/content/drive', force_remount= True)
thisrf_df.to_csv('/content/drive/MyDrive/Aerosol/files/Model/This_Study.csv', index=False)

In [ ]:
choirf_df.to_csv('/content/drive/MyDrive/Aerosol/files/Model/Choi.csv', index=False)

# **Applying model on data**

In [ ]:
drive.mount('/content/drive', force_remount= True)

In [ ]:
this_map = pd.read_csv('/content/drive/MyDrive/Aerosol/files/tropomi_mcd.csv')
choi_map = pd.read_csv('/content/drive/MyDrive/Aerosol/files/myd_tropomi_mcd.csv')

In [ ]:
features = ['LC_IGBP', 'Pct_Agreement', 'CO', 'AER_AI', 'NO2', 'SZA']

X_new = this_map[features]

# Make predictions
new_predictions = model_this.predict(X_new)
new_predictions_4 = model_this_4.predict(X_new)

# Add predictions to new column
this_map['Predicted_Type'] = new_predictions
this_map['Predicted_Type_4'] = new_predictions_4

In [ ]:
this_map

In [ ]:
choi_map.columns

In [ ]:
# Reverse mapping: new (simplified) names → original names
reverse_rename_map = {
    'AOD_550nm': 'AOD_550nm_MYD',
    'Angstrom_Exponent': 'Angstrom_Exponent_MYD',
    'Reflect_412nm': 'Reflect_412nm_MYD',
    'Reflect_470nm': 'Reflect_470nm_MYD',
    'Reflect_660nm': 'Reflect_660nm_MYD',
    'CO': 'CO',
    'AER_AI': 'AER_AI',
    'NO2': 'NO2',
    'SZA': 'SZA',
    'LC_IGBP': 'LC_IGBP',
    'Pct_Agreement': 'Pct_Agreement'
}

choi_map = choi_map.rename(columns=reverse_rename_map)

In [ ]:
choi_map.columns

In [ ]:
features = ['AOD_550nm_MYD', 'Angstrom_Exponent_MYD', 'Reflect_412nm_MYD',
       'Reflect_470nm_MYD', 'Reflect_660nm_MYD', 'LC_IGBP', 'Pct_Agreement', 'CO', 'AER_AI', 'NO2', 'SZA']

X_new = choi_map[features]

# Make predictions
new_predictions = model_choi.predict(X_new)
new_predictions_4 = model_choi_4.predict(X_new)

# Add predictions to new column
choi_map['Predicted_Type'] = new_predictions
choi_map['Predicted_Type_4'] = new_predictions_4

In [ ]:
import os

folder_path = '/content/drive/MyDrive/Aerosol/files/Map Plot'

os.makedirs(folder_path, exist_ok=True)

this_map.to_csv(f'{folder_path}/This_Map.csv', index=False)
choi_map.to_csv(f'{folder_path}/Choi_Map.csv', index=False)
